# Week 12 — Wednesday: Time-Series Data

**DATA 202 · Calvin University**

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/12/images/bikeshare.jpg" width="480" alt="A row of red Capital Bikeshare bikes docked at a station on a Washington, D.C. sidewalk.">

Capital Bikeshare lets people in Washington, D.C. rent a bike at one station and return it at another. Every hour, someone has to decide how many bikes to move to which station, and how many mechanics to put on shift. For that they need to know how many bikes people **will** rent.

> 🎯 **How many bikes next hour? Next Monday at 8 am? And how do we test a model that predicts the future?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~6 min | 1 · Two years, hour by hour |
| ~9 min | 2 · A random split peeks at the future (SLO 12B) |
| ~10 min | 3 · Lag features: what just happened (SLO 12B) |
| ~8 min | 4 · Cross-validation in time order (SLO 12B) |
| ~7 min | 5 · How far ahead? |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: H. Fanaee-T & J. Gama (2013), *Progress in Artificial Intelligence* 2, 113–127; Bike Sharing, UCI Machine Learning Repository, CC BY 4.0 (hours with no rentals added back as 0). Photo: MB-one, CC BY-SA 3.0, via Wikimedia Commons (F Street NW, July 2012)</sub>

---
## 1 · Two Years, Hour by Hour · ~6 min

One row = one hour, from January 1, 2011 to December 31, 2012: 17,544 hours.

* `rentals`: bikes rented that hour (what we predict)
* `temp_c`, `humidity`: the weather that hour
* `weather`: 1 = clear, 2 = mist or clouds, 3 = light rain or snow, 4 = heavy rain or snow
* `holiday`: 1 = a public holiday

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, KFold, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

bikes = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/bikes.csv", parse_dates=["time"])
daily = bikes.resample("D", on="time")["rentals"].sum()          # add up the 24 hours of each day
fig = px.line(daily, title="Bikes Rented per Day, 2011–2012", labels={"value": "bikes rented", "time": ""})
fig.update_layout(showlegend=False)
fig.show()
bikes.head()

In [ ]:
week = bikes[(bikes["time"] >= "2012-09-10") & (bikes["time"] < "2012-09-17")]       # Monday to Sunday
fig = px.line(week, x="time", y="rentals", title="One Week, Hour by Hour: September 10–16, 2012",
              labels={"rentals": "bikes rented per hour", "time": ""})
fig.show()

> 📄 **Handout 1 — One week.** Label the commutes and the weekend. 🗣️ What would you predict for **Monday, September 17, at 8 am**? What are you basing it on?

In [ ]:
hours = ["2012-09-10 08:00", "2012-09-16 08:00", "2012-09-17 07:00", "2012-09-17 08:00"]
bikes.set_index("time").loc[hours, ["rentals"]]

<details><summary>Answer</summary>

* **Weekdays:** two peaks, about 8 am and 5–6 pm: commuters. **Weekend:** one wide hump in the afternoon
* Across the two years: summer above winter, and **2012 above 2011** (the system kept growing). A few deep dips: storms (Hurricane Sandy, October 29, 2012: 22 bikes all day)
* Monday the 17th at 8 am: **683**. The same hour a week before: 730. The day before (Sunday) at 8 am: 128. One hour before: 492
* Your guess probably came from the **past**, picked with care: last Monday, not yesterday. That instinct is today's main tool
</details>

**Time-series data:** rows in time order, where each value depends on the ones before it. A time column alone doesn't make it time series; it's a time series when the order carries information.

---
## 2 · A Random Split Peeks at the Future (SLO 12B) · ~9 min

Pull clock features out of `time`, and give a random forest (Week 11) the clock and the weather:

In [ ]:
bikes["hour"] = bikes["time"].dt.hour
bikes["day_of_week"] = bikes["time"].dt.dayofweek           # 0 = Monday ... 6 = Sunday
clock_weather = ["hour", "day_of_week", "holiday", "temp_c", "humidity", "weather"]
y = bikes["rentals"]

forest = RandomForestRegressor(random_state=1, n_jobs=-1)

# Week 6's random split: 25% of the hours, picked at random
X_tr, X_te, y_tr, y_te = train_test_split(bikes[clock_weather], y, test_size=0.25, random_state=1)
forest.fit(X_tr, y_tr)
print(f"random split | test MAE: {mean_absolute_error(y_te, forest.predict(X_te)):.0f} bikes per hour")

# A time split: learn from the past, test on the future (the last 6 months)
train = bikes["time"] < "2012-07-01"
test = ~train
forest.fit(bikes.loc[train, clock_weather], y[train])
print(f"time split   | test MAE: {mean_absolute_error(y[test], forest.predict(bikes.loc[test, clock_weather])):.0f} bikes per hour")

> 💬 **Question:** Same model, same data, same test size. Why do the two scores differ? Which one would the bike company see in real life?

In [ ]:
results = bikes.loc[test, ["time", "rentals"]].assign(predicted=forest.predict(bikes.loc[test, clock_weather]))
fig = px.line(results.resample("D", on="time").sum(), y=["rentals", "predicted"],
              title="Time Split: the Last 6 Months, Bikes per Day", labels={"value": "bikes per day", "time": "", "variable": ""})
fig.show()

> 📄 **Handout 2 — The forecast.** Shade the gap between the forecast and what happened. 🗣️ Why does the forest guess too low?

<details><summary>Answer</summary>

* **Random split: MAE 50.** Each test hour has training hours from the same day around it (7 am and 9 am learned, 8 am tested), with almost the same weather. The forest can recognize that day by its weather and copy its level: the future **leaked** into training. It's like scoring a model on the rows it learned from (Week 6): overfitting stays hidden
* **Time split: MAE 82**, and too low on 170 of the 184 days: 179 bikes per hour on average, against 252 actual
* A tree predicts **averages of training rows**. It learned 2011 and early 2012; then the system grew. A forest can't predict more bikes than it has seen on a similar day
* **Rule:** with time series, the test set must come **after** the training set
</details>

> 💬 **Question:** `hour` is a number from 0 to 23. Is 11 pm far from midnight?

<details><summary>Answer</summary>

* As numbers, 23 and 0 are 23 apart; on the clock, 1 hour apart
* A tree doesn't mind much: it splits hours into ranges, like "before 6 am"
* Distance-based models (kNN) and linear models do mind. The fix: put the hour on a circle, with two columns: `np.sin(2 * np.pi * hour / 24)` and `np.cos(2 * np.pi * hour / 24)`. Then 11 pm and midnight are neighbors
</details>

---
## 3 · Lag Features: What Just Happened (SLO 12B) · ~10 min

A **lag feature** is a past value of the target, moved down into the current row with `shift`.

> 📄 **Handout 3 — Shift by hand.** Fill in `shift(1)` and `shift(-1)` for the first five hours of the data. 🗣️ Which column could a model really have when it makes its prediction?

In [ ]:
bikes["rentals_1h_ago"] = bikes["rentals"].shift(1)
bikes["rentals_24h_ago"] = bikes["rentals"].shift(24)        # same hour yesterday
bikes["rentals_1wk_ago"] = bikes["rentals"].shift(168)       # same hour last week: 7 × 24 hours
bikes[["time", "rentals", "rentals_1h_ago", "rentals_24h_ago"]].head(5)

<details><summary>Answer</summary>

* `shift(1)` moves each value **down** one row: 1 hour ago. The first row has no hour before it, so pandas puts `NaN` there. A lag of 168 hours leaves the whole first week empty, so we drop those rows
* `shift(-1)` moves the **next** hour up into this row: the answer, handed over before the question. A model with it would score almost perfectly, and be useless: at prediction time, that hour hasn't happened
</details>

A **rolling** feature summarizes a recent window. Shift first, so the window ends at the hour **before** the one we predict:

In [ ]:
bikes["mean_last_24h"] = bikes["rentals"].shift(1).rolling(24).mean()     # average of the 24 hours before this one
bikes = bikes.dropna()           # the first week has no lag values

lags = ["rentals_1h_ago", "rentals_24h_ago", "rentals_1wk_ago", "mean_last_24h"]
train = bikes["time"] < "2012-07-01"
test = ~train
y = bikes["rentals"]

> 🎯 **Predict First:** No model at all, just "same as one week ago". Does it beat Part 2's forest (MAE 82)?

In [ ]:
for lag in ["rentals_1h_ago", "rentals_24h_ago", "rentals_1wk_ago"]:
    print(f"no model, just copy {lag:16} | test MAE: {mean_absolute_error(y[test], bikes.loc[test, lag]):.0f}")

forest.fit(bikes.loc[train, clock_weather + lags], y[train])
print(f"forest, clock + weather + lags | test MAE: {mean_absolute_error(y[test], forest.predict(bikes.loc[test, clock_weather + lags])):.0f}")

<details><summary>Answer</summary>

* Copying last week's value: **MAE 66**, better than Part 2's forest (82). The past carries the **level**: last week already had 2012's bigger crowds
* Copying the last hour (85) or yesterday (80) is worse: 8 am isn't like 7 am, and Monday isn't like Sunday
* **Forest with lags: MAE 33**, and on average right on the level (251 predicted, 252 actual). The clock gives the shape of the day; the lags tell where the level is now
* Always compare with a **naive forecast** (copy the past). A model that can't beat it isn't worth running
</details>

In [ ]:
results = bikes.loc[test, ["time", "rentals"]].assign(predicted=forest.predict(bikes.loc[test, clock_weather + lags]))
fig = px.line(results.resample("D", on="time").sum(), y=["rentals", "predicted"],
              title="With Lag Features: the Last 6 Months, Bikes per Day", labels={"value": "bikes per day", "time": "", "variable": ""})
fig.show()

📌 **One split:** one six-month test period, July to December 2012, including a hurricane. Another stretch of time, other numbers.

---
## 4 · Cross-Validation in Time Order (SLO 12B) · ~8 min

Week 7's folds have the same problem as a random split: the model learns from hours **after** (and right next to) the ones it's checked on. `TimeSeriesSplit` always learns from the past and checks on the stretch that comes next:

In [ ]:
X_dev, y_dev = bikes.loc[train], y[train]           # the 18 months before July 2012; the test stays locked
for fold, (learn, check) in enumerate(TimeSeriesSplit(n_splits=5).split(X_dev)):
    print(f"fold {fold + 1}: learn from {X_dev['time'].iloc[learn[0]]:%b %Y} to {X_dev['time'].iloc[learn[-1]]:%b %Y}"
          f" | check {X_dev['time'].iloc[check[0]]:%b %Y} to {X_dev['time'].iloc[check[-1]]:%b %Y}")

> 🎯 **Predict First:** Which folds will promise the smaller error: shuffled or time-ordered? Which will come closer to what the future delivered (Part 3's test: 82 without lags, 33 with them)?

In [ ]:
for features, name in [(clock_weather, "clock + weather"), (clock_weather + lags, "clock + weather + lags")]:
    shuffled = -cross_val_score(forest, X_dev[features], y_dev, cv=KFold(5, shuffle=True, random_state=1),
                                scoring="neg_mean_absolute_error").mean()
    in_order = -cross_val_score(forest, X_dev[features], y_dev, cv=TimeSeriesSplit(n_splits=5),
                                scoring="neg_mean_absolute_error").mean()
    print(f"{name:24} | CV MAE, shuffled folds: {shuffled:.0f} | CV MAE, time-ordered folds: {in_order:.0f}")

<details><summary>Answer</summary>

* **Shuffled folds** promise 40 and 22: as rosy as Part 2's random split
* **Time-ordered folds** say 57 and 36, much closer to what the future delivered (82 and 33)
* Shuffled CV would also pick knobs (Week 7's grid search) that look good only when the model can peek. With time series, use `cv=TimeSeriesSplit(...)` in `cross_val_score` and `GridSearchCV`
* The window **grows** by default. `TimeSeriesSplit(max_train_size=...)` keeps it a fixed size, if old data stops being useful
</details>

---
## 5 · How Far Ahead? · ~7 min

The lags assumed we know the last hour. To plan **tomorrow's** mechanics' shifts, the latest rentals we know are a day old. To forecast $h$ hours ahead, the newest lag must be $h$ hours old:

In [ ]:
rows = []
for h in [1, 2, 3, 6, 12, 24, 48, 72, 168]:          # 168 hours = 1 week
    X_h = bikes[clock_weather].copy()           # (we pretend the weather forecast is perfect)
    X_h["latest_known"] = bikes["rentals"].shift(h)
    X_h["same_hour_last_week"] = bikes["rentals"].shift(168)
    keep = X_h.notna().all(axis=1)
    forest.fit(X_h[train & keep], y[train & keep])
    rows.append({"hours ahead": h, "test MAE": mean_absolute_error(y[test], forest.predict(X_h[test]))})
fig = px.line(pd.DataFrame(rows), x="hours ahead", y="test MAE", markers=True, log_x=True,
              title="Forecasting Further Ahead", labels={"test MAE": "test MAE (bikes per hour)"})
fig.show()

> 📄 **Handout 4 — How far ahead?** Explain the dip at 24 hours. 🗣️ The bike company asks for next week's forecast. What would you tell them along with the number?

<details><summary>Answer</summary>

* MAE climbs from **34** one hour ahead to about **52–54** at 3–12 hours, then levels off near **56** a week ahead, where only last week's values are left
* **The dip at 24 hours (49):** a day ahead, the newest value we know is the **same hour yesterday**, a better clue than 12 hours ago (8 pm for an 8 am forecast)
* A forecast is a number **and** its typical error. A week out, the typical miss is two-thirds bigger than an hour out: say so
* Forecasting many steps by feeding predictions back in as lags makes errors pile up; it doesn't make the future more knowable
</details>

From Thomas Aquinas's distinction between *studiositas* (virtuous curiosity) and *curiositas* (vicious curiosity), W. Bradford Littlejohn lists seven vices of curiosity. One of them:

> **Impertinence:** "seeking to know things in a more certain way than one can know, doing violence to the object of knowledge."

> 💬 **Question:** Where do you see forecasts reported with more certainty than the data can give? What would an honest forecast of next week's bike rentals look like?

<details><summary>Answer</summary>

* A single number for next week hides that its typical miss is 56 bikes an hour. Honest: a range, the error by horizon, and an update as new hours come in
* Elections, markets, sports, "the model says" in the news: a number with no error bar invites people to treat it as fact
* Virtuous curiosity is moved by love, here, for riders who need a bike at 8 am. It plans ahead, and it knows its limits: "you do not know what tomorrow will bring" (James 4:14)
</details>

---
### Wrap-up

1. **Time-series data:** rows in time order, each depending on the ones before
2. Pull features out of the clock (`.dt.hour`, `.dt.dayofweek`); hours wrap around, so linear models and kNN need sin/cos
3. **Lag features** (`shift`) and **rolling** features (`shift(1).rolling(...)`) carry the recent level. Never shift backwards: that leaks the future
4. **Test on the future:** a random split, or shuffled CV, lets the model peek. Use a time split and `TimeSeriesSplit` 📌
5. Beat a naive forecast first. The further ahead, the larger the error: report both